# Macaque MRI Atlas — REGISTRATION ONLY (v4)

Implements pipeline tree nodes **0**, **2.1**, **2.2**, **2.3**.
Every tunable value lives in cell 1 (`0 SETUP AND PARAMETERS`); no later cell assigns one.

| cell | tree node | writes |
|---|---|---|
| 1 | 0.1–0.4 | — |
| 2 | 2.1, 2.2 | `template_db09/*_sq.nii.gz`, `work/reg/` |
| 3 | 2.3 | `work/reg_subjgrid/` |
| 4 | 2.2.6 | — (diagnostic) |


In [ ]:
# =====================================================================================
# 0  SETUP AND PARAMETERS
# =====================================================================================
# Registration half of the pipeline: nodes 0.1-0.4, 2.1, 2.2, 2.3.
# Produces the files the TEST notebook and the Experiment Runner read from Drive:
#   template/template_db09/mri_rb_fullbrain_sq.nii.gz          (2.1)
#   template/template_db09/DB09_labels_full_full_sq.nii.gz     (2.1, + _LUT.csv)
#   work/reg/<SID>_in_template.nii.gz, <SID>_transforms.pkl    (2.1.3)
#   work/reg_subjgrid/{mri,labels}_in_subjgrid.nii.gz, grid.json (2.3)
# EVERY tunable value in this notebook is set in this cell. No later cell assigns one.
# 0.1  mount Drive and install dependencies
from google.colab import drive; drive.mount('/content/drive')
!pip -q install antspyx nibabel numpy scipy matplotlib

import os, glob, json, time, pickle, sys, csv, colorsys, shutil, hashlib
import numpy as np
import nibabel as nib
import ants
from pathlib import Path

# =====================================================================================
# 0.4.1  PATHS AND RUN IDENTITY
# =====================================================================================
DRIVE_ROOT = "/content/drive/My Drive/macaque_atlas"
WORK       = f"{DRIVE_ROOT}/work_TESTv5"   # must match TEST / Runner / FULL_PIPELINE
TPL_DIR    = f"{DRIVE_ROOT}/template"
TEMPLATE   = "DB09"
SLICE_AXIS = 1        # coronal. Fixes which axis 2.3 gives the acquired slice thickness to.
LR_AXIS    = 0        # after canonical reorient, axis 0 = R/L. Used by the 2.1 mirror.

# =====================================================================================
# 0.4.2  SUBJECTS                                                                  [1.2.1]
# =====================================================================================
# Single source of truth for subject ids and paths. Must be byte-identical to the same
# block in TEST, FULL_PIPELINE and the Experiment Runner: the warped products written here
# are keyed by these ids and read back by those notebooks.
SUBJECT_SCANS = {
    "NHP1": f"{DRIVE_ROOT}/scans/nifti_out/NHP1/NHP1_scan9_reco1_zfix_skullstripped_cropped_final.nii.gz",
    "NHP2": f"{DRIVE_ROOT}/scans/nifti_out/NHP2/NHP2_scan3_reco1_skullstripped_cropped_final_trim.nii.gz",
}

# =====================================================================================
# 0.4.3  REGISTRATION                                                          [2.1 - 2.3]
# =====================================================================================
REBUILD_TEMPLATE   = False       # 2.1  True = rebuild the template files even if present
FORCE_REDO         = False       # 2.1.2  True = re-register every subject, ignoring cache
REG_TRANSFORM      = "Affine"    # 2.1.2.2  "Affine" (12 dof) | "TRSAA" | "SyN"
REG_METRIC         = "mattes"    # 2.1.2.3  MMI
NCC_GATE           = 0.5         # 2.2.2  per-subject pass threshold
PARCELLATION_LEVEL = "full"      # 2.1  label granularity; sets the label filename

# 2.3.1  working-grid voxel size on the TEMPLATE axes. The SLICE_AXIS entry is the acquired
# slice thickness; the other two are the template's own in-plane spacing.
TARGET_ZOOMS          = (0.15, 0.75, 0.15)
TPL_SLICES_TO_CONVERT = (60, 140, 249)   # 2.3.5  template indices to print conversions for

# =====================================================================================
# 0.4.8  DIAGNOSTICS                                                               [2.2.6]
# =====================================================================================
BG_THRESHOLD = 0.0    # 2.2.6.3  voxels > this count as brain (scans are skull-stripped)
DET_GAP_MAX  = 0.10   # 2.2.6.4  max |empirical volume scale - affine determinant| / det
OBLIQUITY_WARN = True # 2.2.6.5  WARN when one acquired plane spreads over more template
                      #          slices than the geometric factor says it should

# =====================================================================================
# 0.2  MODULE AND TEMPLATE INPUTS
# =====================================================================================
os.makedirs(TPL_DIR, exist_ok=True)


# 0.3  locate a template input on Drive
def _find_one(pattern, what="file"):
    """First path matching pattern; raises if nothing matches. [0.3]"""
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matched ({what}): {pattern}")
    return hits[0]


# 0.3.1  template inputs (DB09 MRI, labels, colour/name JSONs)
assert TEMPLATE == "DB09", "This registration-only notebook targets the DB09 template."
DB09_DIR        = f"{TPL_DIR}/template_db09"
TEMPLATE_MRI    = _find_one(f"{DB09_DIR}/**/mri_rb.nii.gz",   "DB09 MRI")
DB09_LABELS_RAW = _find_one(f"{DB09_DIR}/**/atlas_rb.nii.gz", "DB09 labels")
DB09_RGB2ACR    = _find_one(f"{DB09_DIR}/**/rgb2acr.json",    "DB09 rgb2acr")
DB09_ACR2FULL   = _find_one(f"{DB09_DIR}/**/acr2full.json",   "DB09 acr2full")
TEMPLATE_T1     = TEMPLATE_MRI    # replaced by the full-brain squared MRI in 2.1

# 2.1 output names. The TEST notebook globs these exact filenames.
TEMPLATE_MRI_SQ = f"{DB09_DIR}/mri_rb_fullbrain_sq.nii.gz"
TEMPLATE_LABELS = f"{DB09_DIR}/DB09_labels_{PARCELLATION_LEVEL}_full_sq.nii.gz"
TEMPLATE_LUT    = TEMPLATE_LABELS.replace(".nii.gz", "_LUT.csv")

# 2.3 output names.
SUBJ_DIR        = f"{WORK}/reg_subjgrid"
SUBJGRID_MRI    = f"{SUBJ_DIR}/mri_in_subjgrid.nii.gz"
SUBJGRID_LABELS = f"{SUBJ_DIR}/labels_in_subjgrid.nii.gz"
SUBJGRID_LUT    = SUBJGRID_LABELS.replace(".nii.gz", "_LUT.csv")
SUBJGRID_JSON   = f"{SUBJ_DIR}/grid.json"

# =====================================================================================
# 0.3.3  SCRATCH TREE AND STATE HELPERS
# =====================================================================================
# 0.3.3  per-run scratch tree
for d in ["reg", "reg_subjgrid", "qc", "state"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)


def save_state(name, obj):
    """Persist one pipeline object under work/state/. [0.3.3]"""
    with open(f"{WORK}/state/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)


def load_state(name):
    """Read back an object written by save_state. [0.3.3]"""
    with open(f"{WORK}/state/{name}.pkl", "rb") as f:
        return pickle.load(f)


def report_outputs(cell_name, files=(), state_keys=(), checks=()):
    """Print the files, state keys and gate results a cell produced. [0.3.3]"""
    print(f"\n=== {cell_name}: outputs ===")
    for f in files:
        ok = os.path.exists(f); sz = os.path.getsize(f) if ok else 0
        print(f"  {'OK ' if ok and sz>0 else '!! '}{f}  ({sz} bytes)")
    for k in state_keys:
        p = f"{WORK}/state/{k}.pkl"
        print(f"  {'OK ' if os.path.exists(p) else '!! '}state/{k}.pkl")
    for label, okc, hint in checks:
        print(f"  {'PASS' if okc else 'WARN'}: {label}" + ("" if okc else f"  -> {hint}"))
    print("=" * (len(cell_name) + 16))


print("Setup done. Template =", TEMPLATE)
print("  WORK        :", WORK)
print("  DB09 inputs :")
for p in (TEMPLATE_MRI, DB09_LABELS_RAW, DB09_RGB2ACR, DB09_ACR2FULL):
    print("    ", p)
print(f"  subjects    : {list(SUBJECT_SCANS)}")
print(f"  transform   : {REG_TRANSFORM} / {REG_METRIC}   NCC gate {NCC_GATE}")
print(f"  target grid : {TARGET_ZOOMS} mm  (slice axis {SLICE_AXIS})")


In [ ]:
# =====================================================================================
# 2.1  AFFINE REGISTRATION: build the DB09 template, register every subject
#
# 2.1    build full-brain squared MRI + label atlas + LUT (cached)
# 2.1.1  confirm all files exist
# 2.1.2  estimate transform per subject (coarse-to-fine rigid+affine, MMI)
# 2.1.3  emit registration products on the template grid
# 2.1.4  reject a cache built against a different template
# 2.2    score and gate (NCC, NMI)
# =====================================================================================
# 0.3.1  mirror the DB09 hemisphere into a full brain
def _build_full_brain(vol, affine, lr_axis=0):
    """Mirror the DB09 hemisphere across its medial (x~0) face -> full symmetric brain."""
    x0 = affine[0,3]; dx = affine[0,0]; n = vol.shape[lr_axis]
    x_start, x_end = x0, x0 + dx*(n-1)
    at_high_idx = abs(x_end) < abs(x_start)        # medial (x~0) face at the high index?
    flipped = np.flip(vol, axis=lr_axis)
    if at_high_idx:
        full = np.concatenate([vol, flipped], axis=lr_axis); new_x0 = x0
    else:
        full = np.concatenate([flipped, vol], axis=lr_axis); new_x0 = x0 - dx*n
    new_aff = affine.copy(); new_aff[0,3] = new_x0
    return full, new_aff

# 0.3.1  pad in-plane to a square grid (precondition of 6.3.1)
def _pad_square_inplane(vol, slice_axis, affine):
    inplane = [a for a in range(3) if a != slice_axis]
    target  = max(vol.shape[a] for a in inplane)
    pad = [[0,0],[0,0],[0,0]]; shift = np.zeros(3)
    for a in inplane:
        diff = target - vol.shape[a]; before = diff//2
        pad[a] = [before, diff-before]; shift[a] = before
    out = np.pad(vol, pad, mode="constant", constant_values=0)
    new_aff = affine.copy(); new_aff[:3,3] = affine[:3,3] - affine[:3,:3] @ shift
    return out, new_aff

# 0.3.2  build combined_lut (id -> abbreviation / name / rgb / kind)
def _build_lut_from_jsons(rgb2acr, acr2full):
    lut = {}
    for idx, (hexcol, acr) in enumerate(rgb2acr.items()):
        if acr == "[-]" or hexcol.upper() == "000000":
            continue
        full = acr2full.get(acr, "")
        if (not acr) or acr.startswith("["):
            abbrev = acr if acr else f"id{idx}"
            name   = full if full else (acr if acr else f"unnamed_{idx}")
        else:
            abbrev = acr; name = full if full else acr
        lut[idx] = {"abbrev": abbrev, "name": name, "rgb_hex": "#" + hexcol.upper()}
    return lut

# 0.3.1  write the full-brain squared template MRI, labels and LUT
def build_db09_template():
    have_all = all(os.path.exists(p) for p in (TEMPLATE_MRI_SQ, TEMPLATE_LABELS, TEMPLATE_LUT))
    if have_all and not REBUILD_TEMPLATE:
        print("template: cached full-brain squared files present, skipping rebuild.")
        return
    print("template: building full-brain squared DB09 (mirror -> square -> save) ...")
    lab_img = nib.as_closest_canonical(nib.load(DB09_LABELS_RAW))
    mri_img = nib.as_closest_canonical(nib.load(TEMPLATE_MRI))
    merged  = np.asanyarray(lab_img.dataobj).astype(np.int32); ref_affine = lab_img.affine.copy()
    mri_np  = np.asanyarray(mri_img.dataobj).astype(np.float32); mri_aff = mri_img.affine.copy()

    merged, ref_affine = _build_full_brain(merged, ref_affine, LR_AXIS)
    mri_np, mri_aff    = _build_full_brain(mri_np, mri_aff, LR_AXIS)
    merged, ref_affine = _pad_square_inplane(merged, SLICE_AXIS, ref_affine)
    mri_np, mri_aff    = _pad_square_inplane(mri_np, SLICE_AXIS, mri_aff)

    nib.save(nib.Nifti1Image(mri_np, mri_aff), TEMPLATE_MRI_SQ)
    nib.save(nib.Nifti1Image(merged.astype(np.int32), ref_affine), TEMPLATE_LABELS)

    with open(DB09_RGB2ACR)  as f: rgb2acr  = json.load(f)
    with open(DB09_ACR2FULL) as f: acr2full = json.load(f)
    combined = _build_lut_from_jsons(rgb2acr, acr2full)
    def _det_hex(rid):
        h=(int(rid)*0.61803398875)%1.0; r,g,b=colorsys.hsv_to_rgb(h,0.65,0.92)
        return "#{:02x}{:02x}{:02x}".format(round(r*255),round(g*255),round(b*255))
    with open(TEMPLATE_LUT,"w",newline="") as f:
        w=csv.writer(f); w.writerow(["id","abbreviation","name","color_hex","type"])
        for k in sorted(combined):
            e=combined[k]; nm=e["name"]
            kind=("white_matter" if "white matter" in nm.lower()
                  else "ventricle" if "ventricle" in nm.lower() else "gray/other")
            w.writerow([k,e["abbrev"],nm,e.get("rgb_hex") or _det_hex(k),kind])

    # symmetry sanity (a broken mirror shows up here, not 3 cells later)
    n=merged.shape[LR_AXIS]; flip=np.flip(merged,axis=LR_AXIS); lab=merged>0
    labm=float((merged[lab]==flip[lab]).mean()) if lab.any() else float("nan")
    print(f"  full-brain build: shape={merged.shape}, labeled-mirror match={labm*100:.2f}% (want ~100%)")
    print(f"  grid match MRI<->labels: {merged.shape[:3]==mri_np.shape[:3]}")

build_db09_template()
TEMPLATE_T1 = TEMPLATE_MRI_SQ          # register against the full-brain squared MRI
print("  registration fixed image:", TEMPLATE_T1)

# (B) AFFINE REGISTRATION  (cache fixed: transforms persisted to Drive)
from tqdm.auto import tqdm

# 2.1.3.1  forward and inverse transforms recorded on disk
def _persist_transforms(sid, tlist, kind):
    """Copy ANTs transform files out of /tmp into work/reg/ so they survive sessions."""
    out = []
    for i, p in enumerate(tlist):
        ext = os.path.splitext(p)[1] or ".mat"
        dst = f"{WORK}/reg/{sid}_{kind}_{i}{ext}"
        if os.path.abspath(p) != os.path.abspath(dst):
            shutil.copy(p, dst)
        out.append(dst)
    return out

# 2.2.1 / 2.2.3  NCC and NMI between warped subject and template
def registration_metrics(warped_path, template_path=None):
    template_path = template_path or TEMPLATE_T1
    f = ants.image_read(template_path); m = ants.image_read(warped_path)
    fa, ma = f.numpy().ravel(), m.numpy().ravel()
    a = (fa - fa.mean())/(fa.std()+1e-9); b = (ma - ma.mean())/(ma.std()+1e-9)
    # 2.2.1  normalized cross-correlation
    ncc = float(np.mean(a*b))
    # 2.2.3  NMI from a foreground-masked joint histogram
    fg = (fa > 0) | (ma > 0)
    hist,_,_ = np.histogram2d(fa[fg], ma[fg], bins=64)
    pxy = hist/(hist.sum()+1e-9); px = pxy.sum(1); py = pxy.sum(0)
    Hx = -np.sum(px[px>0]*np.log(px[px>0])); Hy = -np.sum(py[py>0]*np.log(py[py>0]))
    Hxy = -np.sum(pxy[pxy>0]*np.log(pxy[pxy>0]))
    return {"ncc": ncc, "nmi": float((Hx+Hy)/(Hxy+1e-9))}

# 2.1.4.1  cache key of the template being registered to
def _template_fp():
    """Identity of the template being registered to. [2.1.4.1]"""
    im = nib.load(TEMPLATE_MRI_SQ)
    return {"file":   os.path.basename(TEMPLATE_MRI_SQ),
            "shape":  tuple(int(s) for s in im.shape[:3]),
            "zooms":  tuple(round(float(z), 5) for z in im.header.get_zooms()[:3]),
            "affine": hashlib.md5(np.round(im.affine, 6).tobytes()).hexdigest(),
            "xform":  REG_TRANSFORM}

# 2.1  affine transform, subject -> template
def register_subjects(transform=REG_TRANSFORM):
    # header-only sanity load of every scan before doing any slow work
    # 2.1.1  confirm every scan exists and its header loads
    for sid, p in SUBJECT_SCANS.items():
        if not (os.path.exists(p) and os.path.getsize(p) > 0):
            raise FileNotFoundError(f"{sid}: scan not found or empty: {p}")
        nib.load(p)
    fp = _template_fp()
    fixed = ants.image_read(TEMPLATE_T1); regs = {}
    bar = tqdm(SUBJECT_SCANS.items(), desc="register", unit="subj")
    for sid, brain_path in bar:
        warped = f"{WORK}/reg/{sid}_in_template.nii.gz"
        tfm    = f"{WORK}/reg/{sid}_transforms.pkl"
        # cache valid only if warped + pkl + every referenced transform file all exist
        valid = (not FORCE_REDO) and os.path.exists(warped) and os.path.exists(tfm)
        if valid:
            with open(tfm,"rb") as f: saved = pickle.load(f)
            valid = (saved.get("template_fp") == fp                    # 2.1.4.2
                     and all(os.path.exists(p) for p in saved["fwd"] + saved["inv"]))
            if not valid:
                print(f"  {sid}: cache REJECTED (template or transform files changed)")
        if valid:
            regs[sid] = {**saved, "warped": warped, "seconds": 0.0, "cached": True}
            bar.set_postfix_str(f"{sid}: cached"); continue
        bar.set_postfix_str(f"{sid}: registering")
        t0 = time.time()
        # 2.1.2  estimate the transform per subject: initial alignment (2.1.2.1), coarse-to-fine rigid+affine (2.1.2.2), MMI similarity (2.1.2.3)
        reg = ants.registration(fixed=fixed, moving=ants.image_read(brain_path),
                                type_of_transform=transform, aff_metric=REG_METRIC)
        dt = time.time() - t0
        # 2.1.3  emit registration products on the template grid
        ants.image_write(reg["warpedmovout"], warped)
        fwd = _persist_transforms(sid, reg["fwdtransforms"], "fwd")
        inv = _persist_transforms(sid, reg["invtransforms"], "inv")
        entry = {"fwd": fwd, "inv": inv, "template_fp": fp}
        with open(tfm,"wb") as f: pickle.dump(entry, f)
        ncc = registration_metrics(warped)["ncc"]
        regs[sid] = {**entry, "warped": warped, "seconds": dt, "cached": False}
        bar.set_postfix_str(f"{sid}: {dt:.0f}s NCC={ncc:.2f}")
    # 2.2.5  save transforms, warped scans, elapsed time, cache status
    save_state("regs", regs)
    checks = []
    for sid, r in regs.items():
        ncc = registration_metrics(r["warped"])["ncc"]
        # 2.2.2  per-subject pass / warn against NCC_GATE
        checks.append((f"{sid} template overlap (NCC={ncc:.2f})", ncc > NCC_GATE,
                       "low overlap -> check strip quality & orientation; try TRSAA or SyN"))
    report_outputs("CELL 2 register",
                   files=[TEMPLATE_MRI_SQ, TEMPLATE_LABELS, TEMPLATE_LUT]
                         + [r["warped"] for r in regs.values()],
                   state_keys=["regs"], checks=checks)
    print("\nThese are the perfect inputs the TEST notebook reads from Drive:")
    print("  template MRI :", TEMPLATE_MRI_SQ)
    print("  template lab :", TEMPLATE_LABELS)
    print("  template LUT :", TEMPLATE_LUT)
    for sid, r in regs.items():
        status = "cached" if r["cached"] else f"{r['seconds']:.0f}s"
        print(f"  {sid}: warped={r['warped']}  transforms={WORK}/reg/{sid}_transforms.pkl "
              f"({status})")
    return regs

register_subjects(transform="Affine")


In [ ]:
# =====================================================================================
# 2.3  COLLAPSE TO THE SHARED WORKING GRID
#
# 2.3.1  target voxel size (in-plane kept, through-plane restored on SLICE_AXIS)
# 2.3.2  build the reference grid by resampling the template
# 2.3.3  resample the template labels with an identity transform
# 2.3.4  resample each subject from the ORIGINAL scan through the 2.1 transform
# 2.3.5  record and verify the new grid
#
# Outputs land in work/reg_subjgrid/; the TEST notebook and Runner glob these names.
# =====================================================================================
import os, json, shutil
import numpy as np, nibabel as nib, ants

# Working-grid voxel size on the TEMPLATE axes. 0.75 sits on SLICE_AXIS (=1, coronal);
# the other two are the template's own in-plane spacing.

# 2.3.5  template slice index -> working-grid slice index
def template_slice_to_subjgrid(idx, scale, n_slices=None):
    """Template slice index -> the slice at the same anatomical position on the new grid."""
    j = int(round(idx * scale))
    return max(0, min(j, n_slices - 1)) if n_slices else max(0, j)

# 2.3  collapse to the shared working grid
def resample_to_subject_grid():
    regs = load_state("regs")
    print(f"working grid on template axes: {TARGET_ZOOMS} mm "
          f"(through-plane on SLICE_AXIS={SLICE_AXIS})")

    # the resampled template MRI IS the reference grid everything else is written onto
    tpl = ants.image_read(TEMPLATE_T1)
    # 2.3.2  build the reference grid: 2.3.2.1 resample the template, 2.3.2.2 new count = old count x old spacing / new spacing
    ref = ants.resample_image(tpl, TARGET_ZOOMS, use_voxels=False, interp_type=0)   # 0 = linear
    ants.image_write(ref, SUBJGRID_MRI)

    # 2.3.3.1  template labels onto the reference grid with an identity transform
    lab_rs = ants.apply_transforms(fixed=ref, moving=ants.image_read(TEMPLATE_LABELS),
                                   transformlist=[], interpolator="nearestNeighbor")
    ants.image_write(lab_rs, SUBJGRID_LABELS)
    shutil.copyfile(TEMPLATE_LUT, SUBJGRID_LUT)

    subj = {}
    for sid, r in regs.items():
        dst = f"{SUBJ_DIR}/{sid}_in_subjgrid.nii.gz"
        # 2.3.4.1  each subject from its ORIGINAL image through the 2.1 transform onto the reference grid (one interpolation)
        w = ants.apply_transforms(fixed=ref, moving=ants.image_read(SUBJECT_SCANS[sid]),
                                  transformlist=r["fwd"], interpolator="linear")
        ants.image_write(w, dst)
        subj[sid] = dst

    tz    = tuple(round(float(z), 5) for z in nib.load(TEMPLATE_T1).header.get_zooms()[:3])
    scale = tz[SLICE_AXIS] / TARGET_ZOOMS[SLICE_AXIS]
    meta  = {"target_zooms": list(TARGET_ZOOMS),
             "target_shape": [int(s) for s in ref.shape],
             "template_zooms": list(tz), "template_shape": [int(s) for s in tpl.shape],
             "slice_axis": SLICE_AXIS, "slice_index_scale": scale,
             "mri": SUBJGRID_MRI, "labels": SUBJGRID_LABELS, "lut": SUBJGRID_LUT,
             "subjects": subj}
    with open(SUBJGRID_JSON, "w") as f:
        json.dump(meta, f, indent=2)

    regs_sg = {sid: {**r, "warped": subj[sid], "warped_template_grid": r["warped"]}
               for sid, r in regs.items()}
    save_state("regs_subjgrid", regs_sg)

    # ---- checks -----------------------------------------------------------------
    a = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    b = nib.load(SUBJGRID_LABELS).get_fdata().astype(np.int32)
    ids_a = set(np.unique(a)) - {0}; ids_b = set(np.unique(b)) - {0}
    lost  = sorted(int(v) for v in ids_a - ids_b)
    D0, D1, D2 = b.shape
    thick = {sid: round(float(max(nib.load(p).header.get_zooms()[:3])), 4)
             for sid, p in SUBJECT_SCANS.items()}
    off   = {sid: v for sid, v in thick.items() if abs(v - TARGET_ZOOMS[SLICE_AXIS]) > 1e-3}
    # 2.3.5  record and verify the new grid and registration outputs
    checks = [
        (f"labels and every subject share one grid {b.shape}",
         all(nib.load(p).shape == b.shape for p in subj.values()),
         "resample wrote mismatched grids"),
        (f"regions kept {len(ids_b)}/{len(ids_a)}", len(lost) == 0,
         f"{len(lost)} region(s) thinner than one {TARGET_ZOOMS[SLICE_AXIS]} mm slice: {lost[:12]}"),
        (f"in-plane grid still square ({D0}x{D2})", D0 == D2, "M3C needs nRow == nCol"),
        (f"acquired slice thickness == grid through-plane ({TARGET_ZOOMS[SLICE_AXIS]} mm)",
         not off, f"acquired {off} mm -- TARGET_ZOOMS no longer matches the scans"),
    ]
    for sid, p in subj.items():
        ncc = registration_metrics(p, SUBJGRID_MRI)["ncc"]
        checks.append((f"{sid} overlap on the subject grid (NCC={ncc:.2f})", ncc > 0.5,
                       "re-check CELL 2 for this subject"))
    report_outputs("CELL 3 resample -> subject grid",
                   files=[SUBJGRID_MRI, SUBJGRID_LABELS, SUBJGRID_LUT] + list(subj.values()),
                   state_keys=["regs_subjgrid"], checks=checks)

    n_new = b.shape[SLICE_AXIS]
    print(f"\n  template grid : {tuple(int(s) for s in tpl.shape)} @ {tz} mm")
    print(f"  working grid  : {tuple(int(s) for s in ref.shape)} @ {meta['target_zooms']} mm")
    print(f"  SLICE_INDEX conversion: new = round(old x {scale:.4f}), valid 0..{n_new-1}")
    for old in TPL_SLICES_TO_CONVERT:
        print(f"    template slice {old:4d} -> working-grid slice "
              f"{template_slice_to_subjgrid(old, scale, n_new)}")
    print(f"  in-plane {TARGET_ZOOMS[0]:.4f} mm = template in-plane: pixel-denominated fusion "
          f"params (kp_alpha, fit_tol_px, SMOOTH_SIGMA, node_match_max, simplify_tol) keep "
          f"their physical size.")
    return regs_sg

resample_to_subject_grid()


In [ ]:
# =====================================================================================
# 2.2.6  TRANSFORM REPORT   (diagnostic only: writes nothing, changes nothing)
#
# 2.2.6.1  affine decomposed on the TEMPLATE ARRAY AXES
# 2.2.6.2  what one native voxel step becomes in template space, per axis
# 2.2.6.3  occupancy extents of the warped brain
# 2.2.6.4  empirical volume scale vs the affine determinant
# 2.2.6.5  obliquity
# =====================================================================================
import numpy as np, nibabel as nib, ants
from scipy.spatial.transform import Rotation as _Rot

regs     = load_state("regs")
_E_CACHE = {}

# 2.2.6.1  RQ decomposition: per-axis scale and shear on the template array axes
def _rq3(M):
    """RQ decomposition M = K @ Qo, K upper-triangular with positive diagonal, Qo a
    rotation. diag(K) = scale along each axis; off-diagonals = shear."""
    P = np.flipud(M).T
    Qq, Rr = np.linalg.qr(P)
    K  = np.fliplr(np.flipud(Rr.T))
    Qo = np.flipud(Qq.T)
    for i in range(3):
        if K[i, i] < 0:
            K[:, i] *= -1.0
            Qo[i, :] *= -1.0
    if np.linalg.det(Qo) < 0:
        K[:, -1] *= -1.0
        Qo[-1, :] *= -1.0
    return K, Qo

# 2.2.6.3  occupancy extents (per-axis data bounding box and brain volume)
def _extent(path, thr=BG_THRESHOLD):
    """Per-axis bounding box of the non-background data, plus total brain volume."""
    if path in _E_CACHE:
        return _E_CACHE[path]
    img = nib.load(path)
    zm  = [float(z) for z in img.header.get_zooms()[:3]]
    shp = tuple(int(s) for s in img.shape[:3])
    m   = np.asanyarray(img.dataobj) > thr
    rows, nvox = [], int(m.sum())
    for a in (0, 1, 2):
        oa = tuple(b for b in (0, 1, 2) if b != a)
        nz = np.flatnonzero(m.any(axis=oa))
        if nz.size == 0:
            rows.append({"axis": a, "lo": -1, "hi": -1, "n": 0, "bg": shp[a], "mm": 0.0})
        else:
            lo, hi = int(nz.min()), int(nz.max())
            n = hi - lo + 1
            rows.append({"axis": a, "lo": lo, "hi": hi, "n": n,
                         "bg": shp[a] - n, "mm": n * zm[a]})
    del m
    _E_CACHE[path] = {"rows": rows, "zooms": zm, "shape": shp,
                      "vol_mm3": nvox * float(np.prod(zm))}
    return _E_CACHE[path]

def _print_extent(tag, e):
    print(f"    {tag}: shape {e['shape']} spacing {tuple(round(z,4) for z in e['zooms'])}"
          f"  brain volume {e['vol_mm3']/1000.0:.2f} cm3")
    for r in e["rows"]:
        print(f"      axis {r['axis']}: data {r['lo']}..{r['hi']} = {r['n']} slices "
              f"({r['mm']:.2f} mm), background {r['bg']} of {e['shape'][r['axis']]}")

# 2.2.6  transform report
def transform_report(sid):
    print(f"\n================ {sid} ================")
    subj = ants.image_read(SUBJECT_SCANS[sid])
    tpl  = ants.image_read(TEMPLATE_T1)
    tx   = ants.read_transform(regs[sid]["fwd"][0])
    par  = np.asarray(tx.parameters, dtype=float)
    fxp  = np.asarray(tx.fixed_parameters, dtype=float)
    if par.size < 12:
        raise ValueError(f"{sid}: fwd[0] has {par.size} params, not a 3D affine")
    A = par[:9].reshape(3, 3)     # ITK: q_moving = A(p_fixed - c) + c + t
    t = par[9:12]
    c = fxp[:3]
    B = np.linalg.inv(A)          # subject -> template, which is what we want to read

    D_s  = np.asarray(subj.direction, dtype=float).reshape(3, 3)
    D_t  = np.asarray(tpl.direction,  dtype=float).reshape(3, 3)
    sp_s = np.asarray(subj.spacing, dtype=float)
    sp_t = np.asarray(tpl.spacing,  dtype=float)
    Dt_i = np.linalg.inv(D_t)
    # 2.2.6.1  affine decomposed on the template array axes
    Barr = Dt_i @ B @ D_t         # same map, expressed on the template's array axes

    print("  --- AFFINE, subject -> template, on the TEMPLATE ARRAY AXES ---")
    detB = float(np.linalg.det(Barr))
    if detB < 0:
        print("  !! NEGATIVE DETERMINANT: this affine contains a REFLECTION. Stop and "
              "check subject orientation before trusting anything downstream.")
    U, S, Vt = np.linalg.svd(Barr)
    Rp = U @ Vt
    if np.linalg.det(Rp) < 0:
        Vt = Vt.copy(); Vt[-1, :] *= -1.0
        Rp = U @ Vt
    rv    = _Rot.from_matrix(Rp)
    vec   = rv.as_rotvec(degrees=True)
    total = float(np.linalg.norm(vec))
    axis  = vec / (total + 1e-12)
    eul   = rv.as_euler("XYZ", degrees=True)
    K, _  = _rq3(Barr)

    print(f"    volume scale (det)   : {detB:.4f}x   (>1 = subject was ENLARGED)")
    print(f"    principal stretches  : {S[0]:.4f} / {S[1]:.4f} / {S[2]:.4f}"
          f"   anisotropy {S[0]/max(S[2],1e-9):.4f}")
    print(f"    TOTAL ROTATION       : {total:.3f} deg")
    print(f"      about array direction ({axis[0]:+.3f}, {axis[1]:+.3f}, {axis[2]:+.3f})")
    print(f"    euler about axes 0/1/2 (XYZ): {eul[0]:+.3f} / {eul[1]:+.3f} / {eul[2]:+.3f} deg")
    print(f"    per-axis scale (RQ)  : {K[0,0]:.4f} / {K[1,1]:.4f} / {K[2,2]:.4f}")
    print(f"    shear (RQ off-diag)  : {K[0,1]:+.4f}  {K[0,2]:+.4f}  {K[1,2]:+.4f}")
    ctr_s = np.asarray(subj.origin, float) + D_s @ (sp_s * (np.array(subj.shape, float) - 1) / 2.0)
    ctr_t = B @ (ctr_s - c - t) + c
    ctr_0 = np.asarray(tpl.origin, float) + D_t @ (sp_t * (np.array(tpl.shape, float) - 1) / 2.0)
    d_mm  = Dt_i @ (ctr_t - ctr_0)
    print(f"    subject centre lands {np.linalg.norm(d_mm):.2f} mm from template centre: "
          f"({d_mm[0]:+.2f}, {d_mm[1]:+.2f}, {d_mm[2]:+.2f}) mm on axes 0/1/2")

    # 2.2.6.2  per-axis voxel step
    print("  --- ONE SUBJECT VOXEL STEP, MEASURED IN TEMPLATE SPACE ---")
    k_thick, info = int(np.argmax(sp_s)), {}
    for k in range(3):
        step = B @ (D_s[:, k] * sp_s[k])
        L    = float(np.linalg.norm(step))
        u    = step / (L + 1e-12)
        comp = np.abs(Dt_i @ u)
        j    = int(np.argmax(comp))
        tilt = float(np.degrees(np.arccos(min(1.0, float(comp[j])))))
        mark = "  <== THICK AXIS" if k == k_thick else ""
        print(f"    subj axis {k} ({sp_s[k]:.4f} mm) -> {L:.4f} mm, nearest tpl axis {j}, "
              f"tilt {tilt:.2f} deg, = {L/float(sp_t[j]):.3f} tpl voxels{mark}")
        if k == k_thick:
            info = {"subj_axis": k, "tpl_axis": j, "len_mm": L,
                    "factor": L / float(sp_t[j]), "tilt": tilt}

    # 2.2.6.3  occupancy extents
    print("  --- OCCUPANCY (data vs background) ---")
    e_tpl = _extent(TEMPLATE_T1)
    e_nat = _extent(SUBJECT_SCANS[sid])
    e_wrp = _extent(regs[sid]["warped"])
    _print_extent("template ", e_tpl)
    _print_extent("native   ", e_nat)
    _print_extent("warped   ", e_wrp)

    # 2.2.6.4  empirical volume scale vs the affine determinant
    emp_vol = e_wrp["vol_mm3"] / max(e_nat["vol_mm3"], 1e-9)
    print(f"    empirical volume scale (warped brain / native brain) = {emp_vol:.4f}x")
    print(f"    affine determinant                                   = {detB:.4f}x")
    print("    (a few percent apart = interpolation halo; a big gap = something is wrong)")

    jt   = info["tpl_axis"]
    nacq = e_nat["rows"][info["subj_axis"]]["n"]
    nwrp = e_wrp["rows"][jt]["n"]
    print(f"    thick axis: {nacq} acquired slices with data -> {nwrp} template slices "
          f"with data = {nwrp/max(nacq,1):.3f} per acquired slice")
    print(f"    geometric factor said {info['factor']:.3f} -- these two MUST agree")

    # 2.2.6.5  obliquity: how far one acquired plane smears across template slices
    half  = 0.5 * max(e_wrp["rows"][a]["mm"] for a in (0, 1, 2) if a != jt)
    smear = 2.0 * half * np.sin(np.radians(info["tilt"])) / float(sp_t[jt])
    print(f"    OBLIQUITY: tilt {info['tilt']:.2f} deg across {2*half:.1f} mm spreads one "
          f"acquired plane over ~{smear:.1f} template slices")
    if smear <= info["factor"]:
        print("      -> acquired planes are near-parallel to template coronal slices.")
    else:
        print("      -> OBLIQUE: no single template coronal slice equals one acquired slice.")
    # 2.2.6  the FULL dict, so 2.2.6.6 can gate on it instead of re-deriving anything
    info.update({"det": detB, "rot_deg": total, "anisotropy": float(S[0] / max(S[2], 1e-9)),
                 "stretches": [float(v) for v in S], "emp_vol_scale": float(emp_vol),
                 "det_gap": float(abs(emp_vol - detB) / max(abs(detB), 1e-9)),
                 "smear": float(smear), "oblique_ok": bool(smear <= info["factor"]),
                 "centre_offset_mm": float(np.linalg.norm(d_mm)),
                 "n_acquired_slices": int(nacq), "n_warped_slices": int(nwrp)})
    return info

# 2.2.6.6  compact table + PASS/WARN gates, fed from the dicts transform_report returns
def transform_summary(all_info):
    """One row per subject: determinant, total rotation, anisotropy, thick-axis tilt, and the
    gap between the EMPIRICAL volume scale and the affine determinant. [2.2.6.6]"""
    print(f"\n{'subject':<10s} {'det':>8s} {'rot deg':>8s} {'aniso':>7s} {'tilt deg':>9s} "
          f"{'emp/det':>8s} {'gap':>7s}")
    print("  " + "-" * 62)
    for sid, i in all_info.items():
        print(f"{sid:<10s} {i['det']:8.4f} {i['rot_deg']:8.3f} {i['anisotropy']:7.4f} "
              f"{i['tilt']:9.2f} {i['emp_vol_scale']:8.4f} {i['det_gap']:7.4f}")
    checks = []
    for sid, i in all_info.items():
        checks.append((f"{sid}: affine determinant is positive (no reflection)",
                       i["det"] > 0,
                       "a negative determinant means the subject is mirrored -- fix the "
                       "orientation before trusting anything downstream"))
        checks.append((f"{sid}: empirical volume scale matches the determinant "
                       f"({i['emp_vol_scale']:.4f} vs {i['det']:.4f}, gap {i['det_gap']:.4f})",
                       i["det_gap"] <= DET_GAP_MAX,
                       f"gap over DET_GAP_MAX={DET_GAP_MAX}: a few percent is interpolation "
                       f"halo, this much means the transform and the warped image disagree"))
        checks.append((f"{sid}: acquired planes are near-parallel to the template slices "
                       f"(tilt {i['tilt']:.2f} deg, smear ~{i['smear']:.1f} slices)",
                       (not OBLIQUITY_WARN) or i["oblique_ok"],
                       f"OBLIQUE: one acquired plane spreads over ~{i['smear']:.1f} template "
                       f"slices against a geometric factor of {i['factor']:.3f}, so no single "
                       f"template coronal slice equals one acquired slice"))
    report_outputs("2.2.6 transform report", checks=checks)
    return all_info

print("=== TRANSFORM REPORT ===")
_all = {}
for _sid in regs:
    _all[_sid] = transform_report(_sid)

print("\n=== SUMMARY ===")
for _sid, _i in _all.items():
    print(f"  {_sid}: thick subj axis {_i['subj_axis']} -> tpl axis {_i['tpl_axis']}, "
          f"{_i['factor']:.3f} tpl voxels per acquired slice, tilt {_i['tilt']:.2f} deg")
transform_summary(_all)
